In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
from typing import List, Dict
import os
import sys
from urllib.parse import urljoin
from pprint import pprint
import re
from typing import Dict

BASE_PATH = "../../data/RAG"

PROBABILITY_BASE_URL = "https://maplestory.nexon.com/Guide/CashShop/Probability/"

prob_list = {
    "치장성": 1,
    "뷰티": 2,
    "게임": 3,
    "펫": 4,
    "잠재능력 재설정": 5,
    "어빌리티": 6,
    "더 시드 반지 상자": 7,
    "보스 반지 상자": 8,
}

In [3]:
def get_probability_urls():
    headers = {
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) " "Chrome/120.0.0.0")
    }

    response = requests.get(PROBABILITY_BASE_URL, headers=headers, timeout=20)

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    probability_items = []

    links = soup.select(".percent_item_rnb a[href]")

    for link in links:
        name = link.get_text(strip=True)
        href = link["href"]

        full_url = urljoin(PROBABILITY_BASE_URL, href)

        seen_urls = set()

                # 같은 URL은 한 번만 저장
        if full_url in seen_urls:
            continue

        seen_urls.add(full_url)

        probability_items.append({
            "name": name,
            "url": full_url
        })

    return probability_items

In [4]:
probability_items = get_probability_urls()

print("찾은 메뉴 개수:", len(probability_items))

for item in probability_items:
    print(item)

찾은 메뉴 개수: 57
{'name': '치장성', 'url': 'https://maplestory.nexon.com/Guide/CashShop/Probability/RoyalStyle'}
{'name': '로얄스타일', 'url': 'https://maplestory.nexon.com/Guide/CashShop/Probability/RoyalStyle'}
{'name': '마스터피스 레드', 'url': 'https://maplestory.nexon.com/Guide/CashShop/Probability/MasterpieceRed'}
{'name': '마스터피스 블랙', 'url': 'https://maplestory.nexon.com/Guide/CashShop/Probability/MasterpieceBlack'}
{'name': '슈피겔만의 마법모자', 'url': 'https://maplestory.nexon.com/Guide/CashShop/Probability/SpiegelmannMagicHat'}
{'name': '미스틱 컬렉션', 'url': 'https://maplestory.nexon.com/Guide/CashShop/Probability/MysticCollection'}
{'name': '그랜드 컬렉션', 'url': 'https://maplestory.nexon.com/Guide/CashShop/Probability/GrandCollection'}
{'name': '장송의 프리렌 코디 컬렉션', 'url': 'https://maplestory.nexon.com/Guide/OtherProbability/Game/CoordiFrieren'}
{'name': '장송의 프리렌 굿즈 컬렉션', 'url': 'https://maplestory.nexon.com/Guide/OtherProbability/Game/GoodsFrieren'}
{'name': '부티크 기프트', 'url': 'https://maplestory.nexon.com/Guide/C

In [5]:
def normalize_html_table(table):
    """
    HTML table의 rowspan, colspan을 풀어서
    모든 행의 열 개수가 동일한 2차원 리스트로 변환
    """

    rows = []

    # 미래 행에 채워져야 하는 rowspan 값 저장
    # key: (행 번호, 열 번호)
    # value: 셀 내용
    spans = {}

    # 중첩 table의 tr이 섞이지 않도록
    # 현재 table에 직접 속한 tr만 가져오기
    trs = []

    for section in table.find_all(
        ["thead", "tbody", "tfoot"],
        recursive=False
    ):
        trs.extend(
            section.find_all(
                "tr",
                recursive=False
            )
        )

    # tbody 없이 바로 tr이 있는 경우
    trs.extend(
        table.find_all(
            "tr",
            recursive=False
        )
    )

    for row_index, tr in enumerate(trs):

        row = []
        col_index = 0

        # 현재 위치에 이전 rowspan 값이 있으면 채우기
        def fill_rowspan():
            nonlocal col_index

            while (row_index, col_index) in spans:
                row.append(
                    spans[(row_index, col_index)]
                )
                col_index += 1

        fill_rowspan()

        cells = tr.find_all(
            ["th", "td"],
            recursive=False
        )

        for cell in cells:

            # 현재 열이 rowspan으로 예약되어 있으면 먼저 채움
            fill_rowspan()

            value = cell.get_text(
                " ",
                strip=True
            )

            rowspan = int(
                cell.get("rowspan", 1) or 1
            )

            colspan = int(
                cell.get("colspan", 1) or 1
            )

            # colspan만큼 현재 행에 복제
            for offset in range(colspan):

                current_col = (
                    col_index + offset
                )

                row.append(value)

                # rowspan이 있으면 아래 행에도 예약
                for future_row in range(
                    row_index + 1,
                    row_index + rowspan
                ):
                    spans[
                        (future_row, current_col)
                    ] = value

            col_index += colspan

        # 행 마지막에 남은 rowspan 처리
        fill_rowspan()

        rows.append(row)

    # 모든 행의 길이를 동일하게 맞춤
    max_columns = max(
        (len(row) for row in rows),
        default=0
    )

    normalized_rows = []

    for row in rows:

        normalized_row = row + (
            [""] * (
                max_columns - len(row)
            )
        )

        normalized_rows.append(
            normalized_row
        )

    return normalized_rows

In [6]:
def get_probability_page(url: str) -> str:
    headers = {
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) " "Chrome/120.0.0.0")
    }

    try:
        response = requests.get(url, headers=headers, timeout=20)

        response.raise_for_status()

        return response.text

    except requests.exceptions.RequestException as e:
        print(f"페이지 요청 중 오류 발생: {e}")
        return ""

In [7]:
def parse_probability_info(html: str, url: str, name: str) -> Dict:

    soup = BeautifulSoup(html, "html.parser")

    content_area = soup.select_one("#container .contents_wrap")

    if content_area is None:
        print("본문 영역을 찾지 못했습니다.")
        return {}

    # -----------------------------
    # 설명 텍스트
    # em 태그만 가져오기
    # -----------------------------

    descriptions = []

    for em in content_area.select("em"):

        text = em.get_text(" ", strip=True)

        if text:
            descriptions.append(text)

    text_content = "\n".join(descriptions)

    # -----------------------------
    # table 추출
    # -----------------------------

    tables = []

    table_tags = content_area.find_all("table")

    for index, table in enumerate(table_tags, start=1):

        # 해당 테이블 앞쪽 제목 찾기
        heading = table.find_previous(["h2", "h3", "h4"])

        if heading:

            # em 등의 자식 내용은 제외하고
            # heading 자체 텍스트만 사용
            section_title = "".join(
                heading.find_all(string=True, recursive=False)
            ).strip()

        else:
            section_title = ""

        # ★ rowspan / colspan 정규화
        rows = normalize_html_table(table)

        if rows:

            tables.append(
                {"table_index": index, "section_title": section_title, "rows": rows}
            )

    probability_info = {
        "name": name,
        "url": url,
        "text_content": text_content,
        "tables": tables,
    }

    return probability_info

In [13]:
url = (
    "https://maplestory.nexon.com/"
    "Guide/OtherProbability/cube/red"
)

html = get_probability_page(url)

red_cube = parse_probability_info(
    html,
    url,
    "레드 큐브"
)

import pandas as pd

rows = red_cube["tables"][2]["rows"]

df = pd.DataFrame(rows)

display(df)

,0,1,2,3,4,5,6,7,8
0,레드 큐브,레어 등급,레어 등급,에픽 등급,에픽 등급,유니크 등급,유니크 등급,레전드리 등급,레전드리 등급
1,첫 번째 옵션,레어,100.0000%,에픽,100.0000%,유니크,100.0000%,레전드리,100.0000%
2,두 번째 옵션,레어,10.0000%,에픽,10.0000%,유니크,10.0000%,레전드리,10.0000%
3,두 번째 옵션,노멀,90.0000%,레어,90.0000%,에픽,90.0000%,유니크,90.0000%
4,세 번째 옵션,레어,1.0000%,에픽,1.0000%,유니크,1.0000%,레전드리,1.0000%
5,세 번째 옵션,노멀,99.0000%,레어,99.0000%,에픽,99.0000%,유니크,99.0000%


In [9]:
all_probability_items = []

for item in probability_items:

    print("수집 중:", item["name"])

    html = get_probability_page(item["url"])

    if not html:
        print("HTML 요청 실패")
        continue

    probability_info = parse_probability_info(html, item["url"], item["name"])

    if probability_info:
        all_probability_items.append(probability_info)

        print(
            f"{item['name']} 수집 완료 "
            f"- 테이블 "
            f"{len(probability_info['tables'])}개"
        )

print("\n전체 수집 페이지:", len(all_probability_items))

수집 중: 치장성
치장성 수집 완료 - 테이블 1개
수집 중: 로얄스타일
로얄스타일 수집 완료 - 테이블 1개
수집 중: 마스터피스 레드
마스터피스 레드 수집 완료 - 테이블 10개
수집 중: 마스터피스 블랙
마스터피스 블랙 수집 완료 - 테이블 10개
수집 중: 슈피겔만의 마법모자
슈피겔만의 마법모자 수집 완료 - 테이블 4개
수집 중: 미스틱 컬렉션
미스틱 컬렉션 수집 완료 - 테이블 1개
수집 중: 그랜드 컬렉션
그랜드 컬렉션 수집 완료 - 테이블 1개
수집 중: 장송의 프리렌 코디 컬렉션
장송의 프리렌 코디 컬렉션 수집 완료 - 테이블 1개
수집 중: 장송의 프리렌 굿즈 컬렉션
장송의 프리렌 굿즈 컬렉션 수집 완료 - 테이블 1개
수집 중: 부티크 기프트
부티크 기프트 수집 완료 - 테이블 2개
수집 중: 알쏭달쏭 라이딩 상자
알쏭달쏭 라이딩 상자 수집 완료 - 테이블 1개
수집 중: 알쏭달쏭 표정 얼굴장식 상자
알쏭달쏭 표정 얼굴장식 상자 수집 완료 - 테이블 1개
수집 중: 알쏭달쏭 코디 상자
알쏭달쏭 코디 상자 수집 완료 - 테이블 1개
수집 중: 뷰티
뷰티 수집 완료 - 테이블 1개
수집 중: 염색 일반 쿠폰
염색 일반 쿠폰 수집 완료 - 테이블 1개
수집 중: 알쏭달쏭 헤어 상자
알쏭달쏭 헤어 상자 수집 완료 - 테이블 1개
수집 중: 알쏭달쏭 성형 상자
알쏭달쏭 성형 상자 수집 완료 - 테이블 1개
수집 중: 게임
게임 수집 완료 - 테이블 1개
수집 중: 골드 애플
골드 애플 수집 완료 - 테이블 1개
수집 중: 플래티넘 애플
플래티넘 애플 수집 완료 - 테이블 1개
수집 중: 주문서/스크롤
주문서/스크롤 수집 완료 - 테이블 8개
수집 중: 추가 옵션
추가 옵션 수집 완료 - 테이블 4개
수집 중: 의문의 모몽
의문의 모몽 수집 완료 - 테이블 1개
수집 중: 상당한 탐험상자
상당한 탐험상자 수집 완료 - 테이블 3개
수집 중: 위대한 소울
위대한 소울 수집 완료 - 테이블 1개
수집 중: 소울 분해
소울 분해 수집 완료 - 테이블 2개


In [10]:
def save_item_to_json(guides, BASE_PATH, file_name="maple_items.json"):

    file_path = os.path.join(BASE_PATH, file_name)

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(guides, f, ensure_ascii=False, indent=2)

    print("JSON 저장 완료")
    return file_path


save_item_to_json(all_probability_items, BASE_PATH)

JSON 저장 완료


'../../data/RAG\\maple_items.json'